In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Naukri Jobs Dataset

In [ ]:
df = pd.read_csv("/content/marketing_sample_for_naukri_com-jobs__20190701_20190830__30k_data.csv")
print("Shape:", df.shape)
df.head(3)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(subset=["Job Title", "Key Skills", "Functional Area"], inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna(subset=["Key Skills", "Job Title", "Functional Area"], inplace=True)
print("Shape after dropping missing values:", df.shape)

## 2. Clean Text

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # strip any stray HTML
    text = text.replace("|", " ")                 # Key Skills are pipe-separated
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # keep letters/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Combine job title + key skills into a single text field (our "review" equivalent)
df["text"] = df["Job Title"].astype(str) + " " + df["Key Skills"].astype(str)
df["clean_text"] = df["text"].apply(clean_text)

## 3. Build the Target: Functional Area (Top 15 + Other)

In [ ]:
# The raw "Functional Area" column has 70+ categories with a long tail.
# Keep the top 15 and bucket everything else into "Other" (matches the site's own convention).
TOP_N = 15
top_classes = df["Functional Area"].value_counts().head(TOP_N).index.tolist()
df["target"] = df["Functional Area"].where(df["Functional Area"].isin(top_classes), "Other")

le = LabelEncoder()
df["label"] = le.fit_transform(df["target"])
print("Classes:", list(le.classes_))

## 4. Train/Test Split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"]
)

## Prepare Sequences for the Neural Network

### Tokenization
Convert each cleaned job posting into a sequence of integers, where each integer represents a word's rank in the vocabulary (fit **only** on the training set).

### Padding
Neural networks need fixed-length input. We pad/truncate every sequence to `max_len=40` (job titles + skills are much shorter than movie reviews).

In [ ]:
tokenizer = Tokenizer(num_words=8000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=40, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=40, padding="post", truncating="post")

y_train_arr = y_train.values
y_test_arr = y_test.values

## 5. Transformer Encoder Building Blocks

A minimal Transformer text classifier: token + positional embeddings, one self-attention encoder block, then masked average pooling over the sequence before the classification head.

In [ ]:
from tensorflow.keras import layers, Model

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
MAX_LEN = 40
VOCAB_SIZE = 8000


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.maxlen = maxlen
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=self.maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update({
            "maxlen": self.maxlen,
            "vocab_size": self.token_emb.input_dim,
            "embed_dim": self.token_emb.output_dim,
        })
        return config


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate,
        })
        return config

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


class MaskedAveragePooling1D(layers.Layer):
    """Average over the time axis, ignoring padded (id==0) positions."""
    def call(self, inputs, token_ids):
        mask = tf.cast(tf.not_equal(token_ids, 0), tf.float32)
        mask = tf.expand_dims(mask, axis=-1)
        summed = tf.reduce_sum(inputs * mask, axis=1)
        counts = tf.maximum(tf.reduce_sum(mask, axis=1), 1.0)
        return summed / counts

## 6. Build & Train the Transformer Model

In [ ]:
token_ids_input = layers.Input(shape=(MAX_LEN,), dtype="int32", name="token_ids")
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(token_ids_input)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = MaskedAveragePooling1D()(x, token_ids_input)
x = layers.Dropout(0.3)(x)
x = layers.Dense(32, activation="relu")(x)
outputs = layers.Dense(len(le.classes_), activation="softmax")(x)

transformer_model = Model(token_ids_input, outputs)

transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_transformer = transformer_model.fit(
    X_train_pad,
    y_train_arr,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=[early_stop]
)

## 7. Evaluate

In [1]:
y_pred_transformer = np.argmax(transformer_model.predict(X_test_pad), axis=1)

print("Test accuracy:", accuracy_score(y_test_arr, y_pred_transformer))
print(classification_report(y_test_arr, y_pred_transformer, target_names=le.classes_))

Test accuracy: 0.7160

                                                              precision    recall  f1-score   support

        Accounts , Finance , Tax , Company Secretary , Audit       0.76      0.85      0.80       265
                                    Engineering Design , R&D       0.71      0.45      0.55        92
      Financial Services , Banking , Investments , Insurance       0.49      0.41      0.45       133
                      HR , Recruitment , Administration , IR       0.83      0.81      0.82       271
         IT Software - Application Programming , Maintenance       0.74      0.80      0.77      1441
                                     IT Software - ERP , CRM       0.58      0.39      0.47        92
                                         IT Software - Other       0.00      0.00      0.00        83
                                  IT Software - QA & Testing       0.62      0.49      0.55        81
      ITES , BPO , KPO , LPO , Customer Service , Operatio

## 8. Save Artifacts

In [ ]:
# Saved in the native Keras format because the model uses custom layers
transformer_model.save("transformer_model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [ ]:
from google.colab import files

files.download("transformer_model.keras")
files.download("tokenizer.pkl")
files.download("label_encoder.pkl")